# 01 資料載入與前處理

**用途**：載入肺癌發生率 CSV，重新命名欄位，篩選「肺、支氣管及氣管／全性別／各縣市」，建立縣市×年份矩陣並計算縣市間／年份間相關係數。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw/lung_cancer_incidence_1979_2022.csv |
| 輸出 | df、df_filtered、pivot_table_age_std（縣市×年）、corr_matrix_county、corr_matrix_yr |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 96–107（Update 05/29 最終版） |

> 原 cell 106 把 pivot 轉成「年×縣市」再取相關，與下一格的變數名稱相反，已移除；此處矩陣一律為縣市×年（.T.corr() = 縣市間、.corr() = 年份間）。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

# Update 05/29

直接從這裡開始跑。

## Load data

$$
標準化死亡率= {\sum(標準人口各年齡層之加權數×觀察族群各年齡層死亡率)}
$$

<br>
$\text{標準人口} = $ WHO 2000 年世界標準人口

## Data Preprocessing

In [ ]:
# rename columns
df.columns = [
    'Diagnosis_Year', 'Gender', 'City_County', 'Cancer_Type',
    'Age_Standardized_Incidence_Rate', 'Cancer_Incidence_Count',
    'Mean_Age', 'Median_Age', 'Crude_Incidence_Rate'
]
# Display the new column names and a preview of the data
print("New column names:", df.columns.tolist())
display(df.head())

In [ ]:
df_filtered = df[(df['Gender'] == '全') &(df['City_County'] != '全國') & (df['Cancer_Type'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='City_County',
    columns='Diagnosis_Year',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
corr_matrix_yr = pivot_table_age_std.corr()
corr_matrix_county = pivot_table_age_std.T.corr()